In [1]:
import sys
print(sys.executable)

/opt/homebrew/anaconda3/bin/python


In [4]:
import sys
{sys.executable} -m !pip uninstall numpy scikit-surprise surprise -y
{sys.executable} -m !pip install "numpy==1.26.4"
{sys.executable} -m !pip install --no-cache-dir scikit-surprise

SyntaxError: invalid syntax (422731136.py, line 2)

In [1]:
import numpy as np
print(np.__version__)

from surprise import Dataset, Reader, SVD
print("Surprise working successfully")

1.26.4
Surprise working successfully


In [9]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

df = pd.read_csv("../data/cleaned_netflix_ratings.csv")

reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    df[["CustomerID", "MovieID", "Rating"]],
    reader
)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

model = SVD()
model.fit(trainset)

predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 1.1821
MAE:  0.9839


In [10]:
def recommend_for_user(user_id, top_n=10):
    all_movies = df["MovieID"].unique()
    watched_movies = df[df["CustomerID"] == user_id]["MovieID"].unique()
    
    unwatched_movies = [movie for movie in all_movies if movie not in watched_movies]
    
    predictions = []
    
    for movie_id in unwatched_movies:
        pred = model.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))
    
    predictions = sorted(predictions, key=lambda x: x[1], reverse=True)
    top_predictions = predictions[:top_n]
    
    movie_ids = [i[0] for i in top_predictions]
    
    result = df[df["MovieID"].isin(movie_ids)][["MovieID", "Title", "Genre", "Year"]].drop_duplicates()
    
    return result

recommend_for_user(1)

,MovieID,Title,Genre,Year
60,238,Led Zeppelin: The Song Remains the Same,Drama,1976
75,2,Isle of Man TT 2004 Review,Drama,2004
147,243,Pressure Point,Drama,1962
169,461,Nightwalker #1: Midnight Detective,Horror,2000
172,428,Barney: Barney's Colorful World: Live,Documentary,2004
306,14,Nature: Antarctica,Documentary,1982
372,480,Flypaper,Drama,1997
517,34,Ashtanga Yoga: Beginner's Practice with Nicki ...,Drama,2003
728,119,Travel the World by Train: Africa,Documentary,1999
739,424,Happiness,Drama,1998


In [7]:
def hybrid_recommendation(user_id, top_n=10):
    all_movies = df["MovieID"].unique()
    watched_movies = df[df["CustomerID"] == user_id]["MovieID"].unique()
    unwatched_movies = [movie for movie in all_movies if movie not in watched_movies]

    user_data = df[df["CustomerID"] == user_id]
    favorite_genres = user_data.groupby("Genre")["Rating"].mean().to_dict()

    recommendations = []

    for movie_id in unwatched_movies:
        movie_info = df[df["MovieID"] == movie_id].iloc[0]

        predicted_rating = model.predict(user_id, movie_id).est
        genre_score = favorite_genres.get(movie_info["Genre"], 3)
        
        movie_popularity = movie_stats[movie_stats["MovieID"] == movie_id]["popularity_score"].values[0]

        normalized_popularity = movie_popularity / movie_stats["popularity_score"].max()

        final_score = (
            0.5 * predicted_rating +
            0.3 * normalized_popularity +
            0.2 * genre_score
        )

        recommendations.append(
            [
                movie_id,
                movie_info["Title"],
                movie_info["Genre"],
                movie_info["Year"],
                predicted_rating,
                final_score
            ]
        )

    result = pd.DataFrame(
        recommendations,
        columns=[
            "MovieID",
            "Title",
            "Genre",
            "Year",
            "Predicted Rating",
            "Hybrid Score"
        ]
    )

    return result.sort_values("Hybrid Score", ascending=False).head(top_n)

hybrid_recommendation(1)

NameError: name 'movie_stats' is not defined